# ViT Standalone Notebook

This notebook isolates the CIFAR ViT path from `VLM.ipynb`.

## Imports

In [ ]:
import os

import matplotlib.pyplot as plt
import torch

from gpt_standalone import TransformerBlock, LayerNormalization, Linear, GLU, MultiHeadAttention
from vit_standalone import (
    CIFAR10,
    ConfigParametersViT,
    OptimParameters,
    ViT,
    evaluate,
    get_loaders,
)


## Shared Transformer Components

These classes are reused from `gpt_standalone.py`.

In [ ]:
LayerNormalization, Linear, GLU, MultiHeadAttention, TransformerBlock


## Config Classes

In [ ]:
ConfigParametersViT, OptimParameters


## Dataset Class

In [ ]:
CIFAR10


## Main Model Class

In [ ]:
ViT


## Main Helper Functions

In [ ]:
get_loaders, evaluate


## Training Setup

In [ ]:
output_dir = "./outputs/vit"
os.makedirs(output_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
patch_size = 8
patch_dim = 192
num_patches = 16
num_classes = 10

cfg_vit = ConfigParametersViT(
    patch_size=patch_size,
    patch_dim=patch_dim,
    num_patches=num_patches,
    num_classes=num_classes,
    device=device,
)
opt_cfg_vit = OptimParameters()

vit_model = ViT(cfg_vit).to(device)
optimizer_vit = torch.optim.AdamW(
    vit_model.parameters(),
    opt_cfg_vit.lr,
    betas=opt_cfg_vit.betas,
    eps=opt_cfg_vit.eps,
)

train_loader, val_loader = get_loaders(batch_size=cfg_vit.batch_size)
len(train_loader), len(val_loader)


## Training Loop

In [ ]:
vit_model.train()
step = 0
max_steps = 781 * 20
train_loss = []

while step < max_steps:
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        logits, loss = vit_model(x, y)

        optimizer_vit.zero_grad()
        loss.backward()
        optimizer_vit.step()

        train_loss.append(loss.item())
        step += 1

        if step % 500 == 0:
            print(f"Step {step} | Loss {loss.item():.4f}")

        if step >= max_steps:
            break

torch.save(
    {"step": step, "model": vit_model.state_dict(), "train_loss": train_loss},
    os.path.join(output_dir, "vit_final.pt"),
)


## Loss Plot

In [ ]:
window = 100
smoothed = [
    sum(train_loss[i : i + window]) / window
    for i in range(len(train_loss) - window + 1)
]

plt.plot(train_loss, alpha=0.2, label="raw")
plt.plot(range(window - 1, len(train_loss)), smoothed, label="smoothed")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("ViT Training Loss")
plt.legend()
plt.show()


## Evaluation

In [ ]:
val_loss, val_acc = evaluate(vit_model, val_loader, device)
print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
